In [42]:
import json
import time
import random
import hashlib
import base64
import urllib.parse
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

LEAGUE_ID = 47
SEASONS   = ['2021_2022', '2022_2023', '2023_2024', '2024_2025']

EXPECTED_ROUNDS            = 38
EXPECTED_MATCHES_PER_ROUND = 10
EXPECTED_TOTAL             = EXPECTED_ROUNDS * EXPECTED_MATCHES_PER_ROUND  # 380

BASE_DATA_DIR = Path('data')
PROCESSED_DIR = Path('data/processed')

# ── FotMob auth ────────────────────────────────────────────────
_FM_SECRET = (
    "We're no strangers to love\n"
    "You know the rules and so do I\n"
    "A full commitment's what I'm thinking of\n"
    "You wouldn't get this from any other guy\n"
    "I just wanna tell you how I'm feeling\n"
    "Gotta make you understand\n"
    "Never gonna give you up\n"
    "Never gonna let you down\n"
    "Never gonna run around and desert you\n"
    "Never gonna make you cry\n"
    "Never gonna say goodbye\n"
    "Never gonna tell a lie and hurt you"
)

# Client version hash — update if requests start failing
# Check DevTools → Network → matchDetails → x-mas header → decode base64 → read 'foo'
FM_CLIENT_VERSION = "production:dda5f4d07deb53ec94eb7009cbab58a6149c4ac3"

# ── Paste your full browser cookie string here ────────────────
# How to refresh: DevTools → Network → matchDetails request
# → Request Headers → copy full 'cookie' value
# Token typically lasts 24-48 hours
BROWSER_COOKIE = """_gcl_au=1.1.746157376.1773609741; _ga=GA1.1.1412827461.1773609741; turnstile_verified=1.1774665970.f859bbd3bc5287b443af3c28dd6b8677398f31f05742e119d10dc25c2e454b0b; u:location=%7B%22countryCode%22%3A%22US%22%2C%22regionId%22%3A%22MA%22%2C%22ip%22%3A%22127.0.0.1%22%2C%22ccode3%22%3A%22USA_MA%22%7D"""

print("Config loaded ✅")

Config loaded ✅


In [43]:
def make_xmas_token(api_path: str, params: dict = None) -> str:
    """
    Build the x-mas header token FotMob requires.
    Must be regenerated fresh for every single request.
    """
    if params:
        qs   = urllib.parse.urlencode(params)
        path = f"{api_path}?{qs}"
    else:
        path = api_path

    code      = int(time.time() * 1000)
    body      = {"url": path, "code": code, "foo": FM_CLIENT_VERSION}
    raw       = json.dumps(body, separators=(',', ':')) + _FM_SECRET
    signature = hashlib.md5(raw.encode('utf-8')).hexdigest().upper()
    token_obj = {"body": body, "signature": signature}
    return base64.b64encode(
        json.dumps(token_obj, separators=(',', ':')).encode('utf-8')
    ).decode('utf-8')


def make_api_headers(path: str, params: dict = None) -> dict:
    """Build full request headers including fresh x-mas token and browser cookie."""
    return {
        'x-mas'          : make_xmas_token(path, params),
        'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept'         : '*/*',
        'Accept-Encoding': 'gzip, deflate, br',
        'Referer'        : 'https://www.fotmob.com/',
        'sec-fetch-dest' : 'empty',
        'sec-fetch-mode' : 'cors',
        'sec-fetch-site' : 'same-origin',
        'cookie'         : BROWSER_COOKIE,
    }


def fetch_match_api(match_id: int) -> tuple:
    """
    Fetch player stats via /api/data/matchDetails.
    Returns (data, status) where status is:
      'ok'           — playerStats found and non-empty
      'auth_blocked' — response ok but playerStats empty/None
      'failed'       — HTTP error or exception
    """
    path   = '/api/data/matchDetails'
    params = {'matchId': match_id}
    url    = f'https://www.fotmob.com{path}'

    try:
        r = requests.get(
            url,
            params=params,
            headers=make_api_headers(path, params),
            timeout=20
        )

        if r.status_code != 200:
            return None, f'failed_http_{r.status_code}'

        data    = r.json()
        content = data.get('content', {})
        ps      = content.get('playerStats')

        if ps and len(ps) > 0:
            return data, 'ok'
        else:
            return data, 'auth_blocked'

    except Exception as e:
        return None, f'failed_exception_{str(e)[:60]}'


print("Token builder and fetch functions defined ✅")

Token builder and fetch functions defined ✅


In [44]:
CATEGORY_MAP = {
    'Top stats'       : 'top_stats',
    'Attack'          : 'attack',
    'Defence'         : 'defense',
    'Defense'         : 'defense',
    'Passes'          : 'passes',
    'Duels'           : 'duels',
    'Goalkeeping'     : 'goalkeeping',
    'Physical metrics': 'physical_metrics',
}

GK_GOALKEEPING = {
    'saves', 'goals_conceded', 'xgot_faced', 'goals_prevented',
    'diving_save', 'saves_inside_box', 'acted_as_sweeper',
    'punches', 'throws', 'high_claim',
}
GK_PASSING = {'accurate_passes', 'accurate_long_balls', 'passes_into_final_third', 'accurate_crosses'}
GK_DEFENSE = {'recoveries', 'clearances', 'defensive_actions', 'tackles', 'interceptions', 'shot_blocks'}
GK_DUELS   = {'ground_duels_won', 'aerials_won', 'was_fouled', 'fouls_committed', 'aerial_duels_won'}

EXCLUDE_FROM_WIDE = {
    'physical_metrics_distance_covered', 'physical_metrics_topspeed',
    'physical_metrics_running', 'physical_metrics_sprinting',
    'physical_metrics_walking', 'physical_metrics_number_of_sprints',
    'fantasy_points', 'Shotmap',
}

ID_COLS = [
    'match_id', 'round', 'match_date',
    'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name',
    'shirt_number', 'position_id', 'is_goalkeeper',
]


def parse_match_api(data: dict) -> list:
    """
    Parse /api/data/matchDetails response into long-format rows.
    Produces identical schema to original parse_match_page so
    build_wide() works completely unchanged.
    """
    rows    = []
    general = data.get('general', {})
    content = data.get('content', {})
    ps      = content.get('playerStats', {})

    if not ps:
        return rows

    match_ctx = {
        'match_id'  : int(general.get('matchId', 0)),
        'round'     : str(general.get('leagueRoundName', '')),
        'match_date': general.get('matchTimeUTCDate', ''),
        'home_team' : general.get('homeTeam', {}).get('name', ''),
        'away_team' : general.get('awayTeam', {}).get('name', ''),
    }

    for pid_str, pdata in ps.items():
        if not isinstance(pdata, dict):
            continue

        is_gk = pdata.get('isGoalkeeper', False)

        player_ctx = {
            'player_id'    : pdata.get('id', pid_str),
            'player_name'  : pdata.get('name', ''),
            'team_id'      : pdata.get('teamId'),
            'team_name'    : pdata.get('teamName', ''),
            'is_goalkeeper': is_gk,
            'shirt_number' : pdata.get('shirtNumber'),
            'position_id'  : pdata.get('positionId'),
        }

        for block in pdata.get('stats', []):
            if not isinstance(block, dict):
                continue

            raw_title  = block.get('title', '')
            block_key  = block.get('key', '')
            stats_dict = block.get('stats', {})

            if not isinstance(stats_dict, dict):
                continue

            for metric_name, metric_data in stats_dict.items():
                if not isinstance(metric_data, dict):
                    continue

                metric_key = metric_data.get('key', '')
                stat       = metric_data.get('stat', {})
                value      = stat.get('value')
                total      = stat.get('total')
                stat_type  = stat.get('type', '')

                if is_gk and block_key == 'top_stats':
                    if metric_key in GK_GOALKEEPING:
                        category = 'goalkeeping'
                    elif metric_key in GK_PASSING:
                        category = 'passes'
                    elif metric_key in GK_DEFENSE:
                        category = 'defense'
                    elif metric_key in GK_DUELS:
                        category = 'duels'
                    else:
                        category = 'top_stats'
                else:
                    category = CATEGORY_MAP.get(raw_title, block_key)

                rows.append({
                    **match_ctx,
                    **player_ctx,
                    'category'  : category,
                    'metric'    : metric_name,
                    'metric_key': metric_key,
                    'value'     : value,
                    'total'     : total,
                    'stat_type' : stat_type,
                })

    return rows


print("parse_match_api defined ✅")

parse_match_api defined ✅


In [45]:
print("=" * 60)
print("LOADING FIXTURES & BUILDING COVERAGE SUMMARY")
print("=" * 60)

fixtures_summary = {}
coverage_summary = {}

for season in SEASONS:
    fix_path   = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output' / 'fixtures.parquet'
    stats_path = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output' / 'player_stats.parquet'

    if not fix_path.exists():
        print(f"[{season}] ❌ fixtures.parquet not found")
        continue

    fix_df = pd.read_parquet(fix_path)
    fixtures_summary[season] = fix_df

    all_match_ids = set(fix_df[fix_df['finished']]['match_id'].astype(int))

    if not stats_path.exists():
        print(f"[{season}] ❌ player_stats.parquet not found")
        coverage_summary[season] = {
            'total': len(all_match_ids), 'scraped': 0,
            'missing': len(all_match_ids), 'missing_ids': sorted(all_match_ids)
        }
        continue

    stats_df    = pd.read_parquet(stats_path)
    scraped_ids = set(stats_df['match_id'].astype(int).unique())
    missing_ids = sorted(all_match_ids - scraped_ids)

    coverage_summary[season] = {
        'total'      : len(all_match_ids),
        'scraped'    : len(scraped_ids),
        'missing'    : len(missing_ids),
        'missing_ids': missing_ids,
    }

    pct = len(scraped_ids) / len(all_match_ids) * 100
    print(f"[{season}]  {len(scraped_ids)}/{len(all_match_ids)} scraped ({pct:.1f}%)  missing: {len(missing_ids)}")

# ── Summary totals ─────────────────────────────────────────────
total_fin  = sum(v['total']   for v in coverage_summary.values())
total_scr  = sum(v['scraped'] for v in coverage_summary.values())
total_mis  = sum(v['missing'] for v in coverage_summary.values())
print(f"\nTotal: {total_scr}/{total_fin} scraped ({total_scr/total_fin*100:.1f}%)  missing: {total_mis}")

LOADING FIXTURES & BUILDING COVERAGE SUMMARY
[2021_2022]  282/380 scraped (74.2%)  missing: 98
[2022_2023]  264/380 scraped (69.5%)  missing: 116
[2023_2024]  269/381 scraped (70.6%)  missing: 112
[2024_2025]  274/380 scraped (72.1%)  missing: 106

Total: 1089/1521 scraped (71.6%)  missing: 432


In [46]:
print("=" * 60)
print("BUILDING RE-SCRAPE TARGET LIST")
print("=" * 60)

rescrape_targets = []

for season in SEASONS:
    info   = coverage_summary.get(season, {})
    fix_df = fixtures_summary.get(season)

    if not info or not info.get('missing_ids') or fix_df is None:
        print(f"[{season}] ✅ Nothing to re-scrape")
        continue

    missing_df           = fix_df[fix_df['match_id'].astype(int).isin(info['missing_ids'])].copy()
    missing_df['season'] = season
    rescrape_targets.append(missing_df)
    print(f"[{season}] — {len(missing_df)} matches queued")

rescrape_df = pd.concat(rescrape_targets, ignore_index=True)
rescrape_df['match_id'] = rescrape_df['match_id'].astype(int)

print(f"\nTotal to re-scrape : {len(rescrape_df)}")
print(f"\nSample:")
print(rescrape_df[['match_id', 'round', 'home_team', 'away_team', 'season']].head(8).to_string(index=False))

BUILDING RE-SCRAPE TARGET LIST
[2021_2022] — 98 matches queued
[2022_2023] — 116 matches queued
[2023_2024] — 112 matches queued
[2024_2025] — 106 matches queued

Total to re-scrape : 432

Sample:
 match_id round               home_team              away_team    season
  3609934     1       Manchester United           Leeds United 2021_2022
  3609930     1                 Burnley Brighton & Hove Albion 2021_2022
  3609935     1        Newcastle United        West Ham United 2021_2022
  3610023    10         Manchester City         Crystal Palace 2021_2022
  3610031    11  Brighton & Hove Albion       Newcastle United 2021_2022
  3610034    11                 Everton      Tottenham Hotspur 2021_2022
  3610048    12 Wolverhampton Wanderers        West Ham United 2021_2022
  3610043    12         Manchester City                Everton 2021_2022


In [47]:
# ── Jitter settings ───────────────────────────────────────────
# Keep between 3-6s to avoid Cloudflare throttling
# If you start seeing 403s mid-run, increase JITTER_MIN/MAX
JITTER_MIN = 3.0
JITTER_MAX = 6.0

newly_recovered  = []
auth_blocked_ids = []
failed_ids       = []

total = len(rescrape_df)
print(f"Re-scraping {total} missing matches via /api/data/matchDetails...\n")
print("⚠️  Note: Browser cookie expires in ~24-48h. If 403s appear mid-run,")
print("    refresh BROWSER_COOKIE in Cell 1 and re-run from Cell 6.\n")

for i, (_, row) in enumerate(rescrape_df.iterrows(), 1):
    match_id = int(row['match_id'])
    season   = row['season']

    raw_dir  = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'raw'
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_path = raw_dir / f'api_{match_id}.json'

    # Skip if already successfully recovered in a previous run
    if raw_path.exists():
        with open(raw_path) as f:
            cached = json.load(f)
        ps = cached.get('content', {}).get('playerStats', {})
        if ps and len(ps) > 0:
            newly_recovered.append(row.to_dict())
            print(f"  ⏭️  {match_id} ({season} R{row['round']}) — already cached")
            continue

    time.sleep(random.uniform(JITTER_MIN, JITTER_MAX))

    data, status = fetch_match_api(match_id)

    if status == 'ok':
        with open(raw_path, 'w') as f:
            json.dump(data, f)
        newly_recovered.append(row.to_dict())
        tag = '✅'
    elif status == 'auth_blocked':
        auth_blocked_ids.append(match_id)
        tag = '🔒'
    else:
        failed_ids.append({'match_id': match_id, 'status': status})
        tag = '❌'

    if i % 25 == 0 or i == total:
        print(f"  [{i}/{total}]  ✅ {len(newly_recovered)}  🔒 {len(auth_blocked_ids)}  ❌ {len(failed_ids)}")
    else:
        print(f"  {tag} {match_id} ({season} R{row['round']}) — {status}")

print(f"\n{'='*50}")
print(f"  ✅ Recovered    : {len(newly_recovered)}")
print(f"  🔒 Auth blocked : {len(auth_blocked_ids)}")
print(f"  ❌ Failed       : {len(failed_ids)}")

Re-scraping 432 missing matches via /api/data/matchDetails...

⚠️  Note: Browser cookie expires in ~24-48h. If 403s appear mid-run,
    refresh BROWSER_COOKIE in Cell 1 and re-run from Cell 6.

  ✅ 3609934 (2021_2022 R1) — ok
  ⏭️  3609930 (2021_2022 R1) — already cached
  ✅ 3609935 (2021_2022 R1) — ok
  ✅ 3610023 (2021_2022 R10) — ok
  ✅ 3610031 (2021_2022 R11) — ok
  ✅ 3610034 (2021_2022 R11) — ok
  ✅ 3610048 (2021_2022 R12) — ok
  ✅ 3610043 (2021_2022 R12) — ok
  ✅ 3610046 (2021_2022 R12) — ok
  ✅ 3610049 (2021_2022 R13) — ok
  ✅ 3610051 (2021_2022 R13) — ok
  ✅ 3610050 (2021_2022 R13) — ok
  ✅ 3610053 (2021_2022 R13) — ok
  ✅ 3610064 (2021_2022 R14) — ok
  ✅ 3610059 (2021_2022 R14) — ok
  ✅ 3610060 (2021_2022 R14) — ok
  ✅ 3610074 (2021_2022 R15) — ok
  ✅ 3610081 (2021_2022 R16) — ok
  ✅ 3610080 (2021_2022 R16) — ok
  ✅ 3610087 (2021_2022 R16) — ok
  ✅ 3610085 (2021_2022 R16) — ok
  ✅ 3610082 (2021_2022 R16) — ok
  ✅ 3610092 (2021_2022 R17) — ok
  ✅ 3610090 (2021_2022 R17) — ok
  [

In [51]:
recovered_df = pd.DataFrame(newly_recovered)

if recovered_df.empty:
    print("No new matches recovered — nothing to merge.")
else:
    print(f"Parsing and merging {len(recovered_df)} recovered matches...\n")

    for season in SEASONS:
        season_recovered = recovered_df[recovered_df['season'] == season]
        if season_recovered.empty:
            print(f"[{season}] — nothing recovered, skipping")
            continue

        raw_dir    = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'raw'
        stats_path = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output' / 'player_stats.parquet'

        existing_stats = pd.read_parquet(stats_path) if stats_path.exists() else pd.DataFrame()
        before_count   = len(existing_stats)

        new_rows = []
        skipped  = 0

        for _, match_row in season_recovered.iterrows():
            mid      = int(match_row['match_id'])
            raw_path = raw_dir / f'api_{mid}.json'

            if not raw_path.exists():
                skipped += 1
                continue

            with open(raw_path) as f:
                api_data = json.load(f)

            rows = parse_match_api(api_data)
            if rows:
                new_rows.extend(rows)
            else:
                skipped += 1

        if not new_rows:
            print(f"[{season}] No new rows parsed (skipped: {skipped})")
            continue

        new_df               = pd.DataFrame(new_rows)
        new_df['match_id']   = new_df['match_id'].astype(int)
        new_df['round']      = pd.to_numeric(new_df['round'], errors='coerce').astype('Int64')
        new_df['match_date'] = pd.to_datetime(new_df['match_date'], utc=True, errors='coerce')
        new_df['value']      = pd.to_numeric(new_df['value'], errors='coerce')
        new_df['total']      = pd.to_numeric(new_df['total'], errors='coerce')

        # Normalize existing to same dtypes before concat
        if not existing_stats.empty:
            existing_stats['match_id'] = existing_stats['match_id'].astype(int)
            existing_stats['round']    = pd.to_numeric(existing_stats['round'], errors='coerce').astype('Int64')
            existing_stats['value']    = pd.to_numeric(existing_stats['value'], errors='coerce')
            existing_stats['total']    = pd.to_numeric(existing_stats['total'], errors='coerce')

        combined = pd.concat([existing_stats, new_df], ignore_index=True)
        combined = combined.drop_duplicates(
            subset=['match_id', 'player_id', 'category', 'metric_key']
        )

        # Final dtype enforcement before writing
        combined['match_id'] = combined['match_id'].astype(int)
        combined['round']    = pd.to_numeric(combined['round'], errors='coerce').astype(int)
        combined['value']    = pd.to_numeric(combined['value'], errors='coerce')
        combined['total']    = pd.to_numeric(combined['total'], errors='coerce')

        combined.to_parquet(stats_path, index=False, engine='pyarrow')
        combined.to_csv(stats_path.with_suffix('.csv'), index=False)

        print(f"[{season}]  +{len(new_df):,} new rows from {len(season_recovered) - skipped} matches")
        print(f"         before: {before_count:,} → after: {len(combined):,} rows")
        if skipped:
            print(f"         ⚠️  skipped: {skipped} (no raw file or empty stats)")

    print("\n✅ player_stats.parquet updated for all seasons")

Parsing and merging 432 recovered matches...

[2021_2022]  +87,483 new rows from 98 matches
         before: 270,156 → after: 357,639 rows
[2022_2023]  +110,193 new rows from 116 matches
         before: 256,602 → after: 366,795 rows
[2023_2024]  +108,099 new rows from 112 matches
         before: 260,272 → after: 368,371 rows
[2024_2025]  +103,919 new rows from 106 matches
         before: 267,302 → after: 371,221 rows

✅ player_stats.parquet updated for all seasons


In [54]:
import sys
import importlib.util
from pathlib import Path

# Load module from file directly since filename starts with a number
spec   = importlib.util.spec_from_file_location(
    "fotmob_scraper",
    Path("01_fotmob_data_scraper.py")  # update path if scraper is in a subfolder
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

build_wide       = module.build_wide
ID_COLS          = module.ID_COLS
EXCLUDE_FROM_WIDE = module.EXCLUDE_FROM_WIDE

print("build_wide imported from 01_fotmob_data_scraper.py ✅\n")

# ── Rebuild wide parquets ──────────────────────────────────────
print("Rebuilding wide-format parquets...\n")

for season in SEASONS:
    stats_path = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output' / 'player_stats.parquet'
    out_dir    = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output'

    if not stats_path.exists():
        print(f"[{season}] ❌ player_stats.parquet not found — skipping")
        continue

    stats_df    = pd.read_parquet(stats_path)
    outfield_df = build_wide(stats_df, is_gk=False)
    gk_df       = build_wide(stats_df, is_gk=True)

    outfield_df.to_parquet(out_dir / 'outfield_players.parquet', index=False, engine='pyarrow')
    outfield_df.to_csv(out_dir / 'outfield_players.csv',         index=False)
    gk_df.to_parquet(out_dir / 'goalkeepers.parquet',            index=False, engine='pyarrow')
    gk_df.to_csv(out_dir / 'goalkeepers.csv',                    index=False)

    print(f"[{season}]  outfield: {len(outfield_df):,} rows | GK: {len(gk_df):,} rows ✅")

print("\n✅ Wide-format parquets rebuilt for all seasons")

c:\Users\natmaw\anaconda3\envs\mlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


build_wide imported from 01_fotmob_data_scraper.py ✅

Rebuilding wide-format parquets...

[2021_2022]  outfield: 7,812 rows | GK: 760 rows ✅
[2022_2023]  outfield: 7,954 rows | GK: 760 rows ✅
[2023_2024]  outfield: 7,733 rows | GK: 762 rows ✅
[2024_2025]  outfield: 7,598 rows | GK: 760 rows ✅

✅ Wide-format parquets rebuilt for all seasons


In [55]:
print("=" * 60)
print("FINAL COVERAGE VERIFICATION")
print("=" * 60)

total_fixtures = 0
total_scraped  = 0

for season in SEASONS:
    fix_df     = fixtures_summary[season]
    stats_path = BASE_DATA_DIR / str(LEAGUE_ID) / season / 'output' / 'player_stats.parquet'

    if not stats_path.exists():
        print(f"[{season}] ❌ player_stats.parquet not found")
        continue

    stats_df    = pd.read_parquet(stats_path)
    all_ids     = set(fix_df[fix_df['finished']]['match_id'].astype(int))
    scraped_ids = set(stats_df['match_id'].astype(int).unique())
    missing_ids = all_ids - scraped_ids

    pct = len(scraped_ids) / len(all_ids) * 100
    total_fixtures += len(all_ids)
    total_scraped  += len(scraped_ids)

    flag = "✅" if len(missing_ids) == 0 else "⚠️ "
    print(f"{flag} [{season}]  {len(scraped_ids)}/{len(all_ids)} ({pct:.1f}%)  still missing: {len(missing_ids)}")

    if missing_ids:
        print(f"     Missing IDs: {sorted(missing_ids)[:10]}{'...' if len(missing_ids) > 10 else ''}")

overall = total_scraped / total_fixtures * 100
print(f"\nOverall: {total_scraped}/{total_fixtures} ({overall:.1f}%)")

if total_scraped == total_fixtures:
    print("\n🎉 100% coverage achieved!")
    print("→ Next: re-run 02_preprocessing.ipynb then 03_feature_engineering.ipynb")
else:
    remaining = total_fixtures - total_scraped
    print(f"\n{remaining} matches still missing — these are permanently auth-blocked")
    print("→ Acknowledged as data limitation in project report")
    print("→ Next: re-run 02_preprocessing.ipynb then 03_feature_engineering.ipynb")

FINAL COVERAGE VERIFICATION
✅ [2021_2022]  380/380 (100.0%)  still missing: 0
✅ [2022_2023]  380/380 (100.0%)  still missing: 0
✅ [2023_2024]  381/381 (100.0%)  still missing: 0
✅ [2024_2025]  380/380 (100.0%)  still missing: 0

Overall: 1521/1521 (100.0%)

🎉 100% coverage achieved!
→ Next: re-run 02_preprocessing.ipynb then 03_feature_engineering.ipynb
